In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from timm.layers.create_act import create_act_layer

In [2]:
class GDFN(nn.Module):
    def __init__(self, dim, ffn_expansion_factor, bias):
        super(GDFN, self).__init__()

        hidden_features = int(dim*ffn_expansion_factor)

        self.project_in = nn.Conv2d(dim, hidden_features*2, kernel_size=3, stride = 1, padding = 1, bias=bias)

        self.dwconv = nn.Conv2d(hidden_features*2, hidden_features*2, kernel_size=3, stride=1, padding=1, groups=hidden_features*2, bias=bias)

        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x


class Attention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads = 8,
        qkv_bias = True,
        use_rel_pos = False,
        rel_pos_zero_init =  True,
        input_size = None,
    ) -> None:

        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

        self.use_rel_pos = use_rel_pos
        if self.use_rel_pos:
            assert (
                input_size is not None
            ), "Input size must be provided if using relative positional encoding."
            self.rel_pos_h = nn.Parameter(torch.zeros(2 * input_size[0] - 1, head_dim))
            self.rel_pos_w = nn.Parameter(torch.zeros(2 * input_size[1] - 1, head_dim))

    def forward(self, x):
        B, H, W, _ = x.shape
        qkv = self.qkv(x).reshape(B, H * W, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.reshape(3, B * self.num_heads, H * W, -1).unbind(0)
        attn = (q * self.scale) @ k.transpose(-2, -1)
        if self.use_rel_pos:
            attn = add_decomposed_rel_pos(attn, q, self.rel_pos_h, self.rel_pos_w, (H, W), (H, W))
        attn = F.relu(attn)
        x = (attn @ v).view(B, self.num_heads, H, W, -1).permute(0, 2, 3, 1, 4).reshape(B, H, W, -1)

        return self.proj(x)


def window_partition(x, window_size):
    B, H, W, C = x.shape

    pad_h = (window_size - H % window_size) % window_size
    pad_w = (window_size - W % window_size) % window_size
    if pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
    Hp, Wp = H + pad_h, W + pad_w

    x = x.view(B, Hp // window_size, window_size, Wp // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows, (Hp, Wp)


def window_unpartition(windows, window_size, pad_hw, hw):
    Hp, Wp = pad_hw
    H, W = hw
    B = windows.shape[0] // (Hp * Wp // window_size // window_size)
    x = windows.view(B, Hp // window_size, Wp // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, Hp, Wp, -1)

    if Hp > H or Wp > W:
        x = x[:, :H, :W, :].contiguous()
    return x

def get_rel_pos(q_size, k_size, rel_pos):
    max_rel_dist = int(2 * max(q_size, k_size) - 1)
    if rel_pos.shape[0] != max_rel_dist:
        rel_pos_resized = F.interpolate(
            rel_pos.reshape(1, rel_pos.shape[0], -1).permute(0, 2, 1),
            size=max_rel_dist,
            mode="linear",
        )
        rel_pos_resized = rel_pos_resized.reshape(-1, max_rel_dist).permute(1, 0)
    else:
        rel_pos_resized = rel_pos

    # Scale the coords with short length if shapes for q and k are different.
    q_coords = torch.arange(q_size)[:, None] * max(k_size / q_size, 1.0)
    k_coords = torch.arange(k_size)[None, :] * max(q_size / k_size, 1.0)
    relative_coords = (q_coords - k_coords) + (k_size - 1) * max(q_size / k_size, 1.0)

    return rel_pos_resized[relative_coords.long()]


def add_decomposed_rel_pos(attn, q, rel_pos_h, rel_pos_w, q_size, k_size):

    q_h, q_w = q_size
    k_h, k_w = k_size
    Rh = get_rel_pos(q_h, k_h, rel_pos_h)
    Rw = get_rel_pos(q_w, k_w, rel_pos_w)

    B, _, dim = q.shape
    r_q = q.reshape(B, q_h, q_w, dim)
    rel_h = torch.einsum("bhwc,hkc->bhwk", r_q, Rh)
    rel_w = torch.einsum("bhwc,wkc->bhwk", r_q, Rw)

    attn = (
        attn.view(B, q_h, q_w, k_h, k_w) + rel_h[:, :, :, :, None] + rel_w[:, :, :, None, :]
    ).view(B, q_h * q_w, k_h * k_w)

    return attn


class SSA(nn.Module):
    def __init__(self, input_channels, window_size, bias):
        super().__init__()
        self.window_size = window_size
        self.attn  = Attention(dim = input_channels, num_heads = 8)
        self.gdfn  = GDFN(input_channels, ffn_expansion_factor = 2.66, bias = bias)
        self.proje = nn.Conv2d(input_channels, input_channels, kernel_size=1, bias=bias)

    def forward(self, x):
        B, C, H, W = x.shape

        V = x

        x = x.permute(0, 2, 3, 1)

        x, pad_hw = window_partition(x, self.window_size)  # nW*B, window_size, window_size, C

        x = self.attn(x)  # nW*B, window_size*window_size, C

        x = window_unpartition(x, self.window_size, pad_hw, (H, W))

        x = x.permute(0, 3, 1, 2)

        V = self.gdfn(V)

        return self.proje(x + V)


class SCA(nn.Module):
    def __init__(self, input_channels, num_heads, bias):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.initial_proj = nn.Conv2d(input_channels, input_channels * 3, kernel_size = 1, bias = bias)
        self.qkv = nn.Conv2d(input_channels * 3, input_channels * 3, kernel_size = 3, padding = 1, groups = input_channels * 3)
        self.gdfn = GDFN(input_channels, ffn_expansion_factor = 2.66, bias = bias)
        self.proje = nn.Conv2d(input_channels, input_channels, kernel_size=1, bias=bias)

    def forward(self, x):
        b, c, h, w = x.shape
        V = x
        x = self.initial_proj(x)
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim = 1)

        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)

        q = torch.nn.functional.normalize(q, dim=-1)
        k = torch.nn.functional.normalize(k, dim=-1)

        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        output = attn @ v
        output = rearrange(output, 'b head c (h w) -> b (head c) h w', h = h, w = w)
        V = self.gdfn(V)

        return self.proje(output + V)


class DSFN(nn.Module):
    def __init__(self, input_channels, hidden_channels):
        super().__init__()
        self.input_proj = nn.Conv2d(input_channels, hidden_channels, kernel_size = 1)
        self.act_fn = nn.GELU()
        self.dwconv = nn.Conv2d(hidden_channels // 2, hidden_channels // 2, kernel_size = 3, stride = 1, padding = 1, groups = hidden_channels // 2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.c_proj = nn.Conv2d(hidden_channels // 2, hidden_channels // 2, kernel_size = 1)
        self.final_proj = nn.Conv2d(hidden_channels // 2, input_channels, kernel_size =1)


    def forward(self, x):
        x = self.act_fn(self.input_proj(x))
        x_s1, x_s2 = x.chunk(2, dim = 1)
        x_s2 = self.dwconv(x_s2)
        x_s = x_s1 * x_s2

        x_c1, x_c2 = x.chunk(2, dim = 1)
        x_c2 = self.c_proj(self.gap(x_c2))
        x_c = x_c1 * x_c2

        x = x_s + x_c

        return self.final_proj(x)

class LayerNorm2d(nn.Module):
    def __init__(self, num_channels: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        x = self.weight[:, None, None] * x + self.bias[:, None, None]
        return x

class RDSTGBlock(nn.Module):
    def __init__(self, dim, window_size, bias = True):
        super().__init__()
        self.layernorm1 = LayerNorm2d(dim)
        self.ssa        = SSA(dim, window_size, bias)
        self.layernorm2 = LayerNorm2d(dim)
        self.dsfn1      = DSFN(dim, dim * 2)
        self.layernorm3 = LayerNorm2d(dim)
        self.sca        = SCA(dim, window_size, bias)
        self.layernorm4 = LayerNorm2d(dim)
        self.dsfn2      = DSFN(dim, dim * 2)

    def forward(self, x):
      shortcut = x
      x = shortcut + self.ssa(self.layernorm1(x))

      shortcut = x
      x = shortcut + self.dsfn1(self.layernorm2(x))

      shortcut = x
      x = shortcut + self.sca(self.layernorm3(x))

      shortcut = x
      x = shortcut + self.dsfn2(self.layernorm4(x))

      return x

In [3]:
class RDSTG(nn.Module):
  def __init__(self, dim, window_size, num_rdstg_block, bias = True):
    super().__init__()
    layers = [RDSTGBlock(dim = dim, window_size = window_size, bias = bias) for _ in range(num_rdstg_block)]
    self.model = nn.Sequential(*layers)
    self.project = nn.Conv2d(dim, dim, kernel_size = 3, stride = 1, padding = 1, bias = bias)

  def forward(self, x):
    shortcut = x
    x = self.model(x)
    return shortcut + self.project(x)


In [4]:
class Backbone(nn.Module):
  def __init__(self, dim, window_size, num_rdstg_block = 4, num_rdstg =4, bias = True):
    super().__init__()
    layers = [RDSTG(dim, window_size, num_rdstg_block, bias) for _ in range(num_rdstg)]
    self.model = nn.Sequential(*layers)
    self.project = nn.Conv2d(dim, dim, kernel_size = 3, stride = 1, padding = 1, bias = bias)

  def forward(self, x):
    shortcut = x
    x = self.model(x)
    return shortcut + self.project(x)

In [5]:
class ISF(nn.Module):
  def __init__(self, input_channels = 3, dim = 64):
    super().__init__()
    self.initial = nn.Conv2d(input_channels, dim, kernel_size = 3, stride = 2, padding = 1)
    self.act_fn = nn.GELU()
    self.project = nn.Conv2d(dim, dim * 2, kernel_size = 3, stride = 1, padding = 1)

  def forward(self, x):
    x = self.act_fn(self.initial(x))
    x = self.project(x)
    return x

In [6]:
class SqueezeExcite(nn.Module):
    def __init__(self, in_chs, rd_ratio = 0.25, rd_channels = None, act_layer = nn.ReLU, gate_layer = nn.Sigmoid, force_act_layer = None, rd_round_fn = None, device=None, dtype=None):
        dd = {'device': device, 'dtype': dtype}
        super().__init__()
        if rd_channels is None:
            rd_round_fn = rd_round_fn or round
            rd_channels = rd_round_fn(in_chs * rd_ratio)
        act_layer = force_act_layer or act_layer
        self.conv_reduce = nn.Conv2d(in_chs, rd_channels, 1, bias=True, **dd)
        self.act1 = create_act_layer(act_layer, inplace=True)
        self.conv_expand = nn.Conv2d(rd_channels, in_chs, 1, bias=True, **dd)
        self.gate = create_act_layer(gate_layer)

    def forward(self, x):
        x_se = x.mean((2, 3), keepdim=True)
        x_se = self.conv_reduce(x_se)
        x_se = self.act1(x_se)
        x_se = self.conv_expand(x_se)
        return x * self.gate(x_se)

In [7]:
class CFIR(nn.Module):
  def __init__(self, input_channels = 128, output_channels = 3):
    super().__init__()
    self.se       = SqueezeExcite(input_channels)
    self.initial  = nn.Conv2d(input_channels, input_channels, kernel_size = 3, stride = 1, padding = 1)
    self.act_fn   = nn.GELU()
    self.upsample = nn.ConvTranspose2d(input_channels, input_channels, kernel_size = 2, stride = 2)
    self.project  = nn.Conv2d(input_channels, output_channels, kernel_size = 3, stride = 1, padding = 1)

  def forward(self, x):
    x = self.se(x)
    x = self.act_fn(self.initial(x))
    x = self.upsample(x)
    x = self.project(x)
    return x

In [17]:
class MyModel(nn.Module):
  def __init__(self, input_channels, output_channels, dim = 128, num_heads = 8):
    super().__init__()
    self.isf = ISF(input_channels)
    self.backbone = Backbone(dim, num_heads)
    self.cfir = CFIR(dim, output_channels)

  def forward(self, x):
    x = self.isf(x)
    x = self.backbone(x)
    x = self.cfir(x)
    return x

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
model = MyModel(input_channels = 3, output_channels = 3).to(device)

In [20]:
test = torch.randn(1, 3, 128, 128).to(device)

In [21]:
print(model(test).shape)

torch.Size([1, 3, 128, 128])


In [22]:
!pip install torchinfo

In [23]:
from torchinfo import summary

In [24]:
summary(model, input_size=(1, 3, 128, 128))

Layer (type:depth-idx)                                       Output Shape              Param #
MyModel                                                      [1, 3, 128, 128]          --
├─ISF: 1-1                                                   [1, 128, 64, 64]          --
│    └─Conv2d: 2-1                                           [1, 64, 64, 64]           1,792
│    └─GELU: 2-2                                             [1, 64, 64, 64]           --
│    └─Conv2d: 2-3                                           [1, 128, 64, 64]          73,856
├─Backbone: 1-2                                              [1, 128, 64, 64]          --
│    └─Sequential: 2-4                                       [1, 128, 64, 64]          --
│    │    └─RDSTG: 3-1                                       [1, 128, 64, 64]          7,976,032
│    │    └─RDSTG: 3-2                                       [1, 128, 64, 64]          7,976,032
│    │    └─RDSTG: 3-3                                       [1, 128, 64, 